# Sheffield Scrollie — DICOM Viewer

Slice-by-slice viewer for the **sheffeld** dataset (augmented lower-limb MRI + segmentations).

**Data structure:**
```
sheffeld/
  20440164/   ← Aug_N.dcm                  (69 augmented greyscale 3D images)
  20440203/   ← Aug_N_segmentations.dcm     (69 paired label masks, DICOM SEG format)
```

The segmentation files are in **DICOM SEG** format: each file contains one binary frame
per (segment × slice), and the label integer is stored in `PerFrameFunctionalGroupsSequence`,
not directly in the pixel values (which are 0/255).

**Label map** (from `20440203/read_me.txt`) — 37 muscles, IDs 1–37, alphabetical order:
```
 1 adductor brevis          14 gluteus medius          27 rectus femoris
 2 adductor longus          15 gluteus minimus         28 sartorius
 3 adductor magnus          16 gracilis                29 semimembranosus
 4 biceps fem. caput breve  17 iliacus                 30 semitendinosus
 5 biceps fem. caput longum 18 obturator externus      31 soleus
 6 extensor digitorum long. 19 obturator internus      32 tensor fasciae latae
 7 extensor hallucis long.  20 pectineus               33 tibialis anterior
 8 flexor digitorum long.   21 peroneus brevis         34 tibialis posterior
 9 flexor hallucis long.    22 peroneus longus         35 vastus intermedius
10 gastrocnemius lateralis  23 piriformis              36 vastus lateralis
11 gastrocnemius medialis   24 popliteus               37 vastus medialis
12 gemellus superior        25 psoas
13 gluteus maximus          26 quadratus femoris
```

**How to use:**
1. Run all cells.
2. Pick an augmentation index (Aug_1 … Aug_69).
3. Toggle **Show Seg** to overlay the segmentation labels.
4. Drag the **Slice** slider.

In [ ]:
import os
import glob
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, VBox, HBox, Checkbox, Output
from IPython.display import display

try:
    import pydicom
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pydicom'])
    import pydicom

print(f'pydicom {pydicom.__version__}')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATA_ROOT = r'C:\Projects\dissector\eval_notebooks\sheffeld'
IMG_DIR   = os.path.join(DATA_ROOT, '20440164')   # Aug_N.dcm
SEG_DIR   = os.path.join(DATA_ROOT, '20440203')   # Aug_N_segmentations.dcm

_seg_files = sorted(
    glob.glob(os.path.join(SEG_DIR, 'Aug_*_segmentations.dcm')),
    key=lambda p: int(re.search(r'Aug_(\d+)_segmentations', p).group(1)),
)
INDICES = [re.search(r'Aug_(\d+)_segmentations', f).group(1) for f in _seg_files]

print(f'Images dir  : {IMG_DIR}  (exists={os.path.isdir(IMG_DIR)})')
print(f'Segs dir    : {SEG_DIR}  (exists={os.path.isdir(SEG_DIR)})')
print(f'Aug indices : {len(INDICES)} found  '
      f'(first={INDICES[0] if INDICES else "—"}, last={INDICES[-1] if INDICES else "—"})')

In [ ]:
# ── Inspect one segmentation DICOM — run once to confirm DICOM SEG format ─────
_ds = pydicom.dcmread(os.path.join(SEG_DIR, 'Aug_1_segmentations.dcm'))

print('SOP Class UID :', getattr(_ds, 'SOPClassUID', 'n/a'))
print('Modality      :', getattr(_ds, 'Modality', 'n/a'))
print('Rows x Cols   :', getattr(_ds, 'Rows', '?'), 'x', getattr(_ds, 'Columns', '?'))
print('NumberOfFrames:', getattr(_ds, 'NumberOfFrames', 'n/a'))

_arr = _ds.pixel_array
print(f'pixel_array   : shape={_arr.shape}  dtype={_arr.dtype}  '
      f'unique={sorted(set(_arr.flat))[:10]}')

if hasattr(_ds, 'SegmentSequence'):
    print(f'\nSegmentSequence ({len(_ds.SegmentSequence)} segments):')
    for seg in list(_ds.SegmentSequence)[:6]:
        print(f'  #{int(seg.SegmentNumber):3d}  {getattr(seg, "SegmentLabel", "?")}')
    if len(_ds.SegmentSequence) > 6:
        print(f'  ... ({len(_ds.SegmentSequence)} total)')

if hasattr(_ds, 'PerFrameFunctionalGroupsSequence'):
    _pf = _ds.PerFrameFunctionalGroupsSequence
    print(f'\nPerFrameFunctionalGroupsSequence: {len(_pf)} frames')
    _f0 = _pf[0]
    if hasattr(_f0, 'SegmentIdentificationSequence'):
        print('  Frame 0 seg# :', int(_f0.SegmentIdentificationSequence[0].ReferencedSegmentNumber))
    if hasattr(_f0, 'PlanePositionSequence'):
        print('  Frame 0 z    :', _f0.PlanePositionSequence[0].ImagePositionPatient[2])

In [ ]:
# ── Label map (from 20440203/read_me.txt) ─────────────────────────────────────
# Labels 1–37 alphabetically. Pixel values in the DICOM are spread evenly
# across 0–255:  pixel_value = round(label_index × 255 / 37)
# So 0→background, 7→label 1, 14→label 2, 34→label 5, 255→label 37.
SHEFFIELD_LABELS = {
     1: 'adductor brevis',
     2: 'adductor longus',
     3: 'adductor magnus',
     4: 'biceps femoris caput breve',
     5: 'biceps femoris caput longum',
     6: 'extensor digitorum longus',
     7: 'extensor hallucis longus',
     8: 'flexor digitorum longus',
     9: 'flexor hallucis longus',
    10: 'gastrocnemius lateralis',
    11: 'gastrocnemius medialis',
    12: 'gemellus superior',
    13: 'gluteus maximus',
    14: 'gluteus medius',
    15: 'gluteus minimus',
    16: 'gracilis',
    17: 'iliacus',
    18: 'obturator externus',
    19: 'obturator internus',
    20: 'pectineus',
    21: 'peroneus brevis',
    22: 'peroneus longus',
    23: 'piriformis',
    24: 'popliteus',
    25: 'psoas',
    26: 'quadratus femoris',
    27: 'rectus femoris',
    28: 'sartorius',
    29: 'semimembranosus',
    30: 'semitendinosus',
    31: 'soleus',
    32: 'tensor fasciae latae',
    33: 'tibialis anterior',
    34: 'tibialis posterior',
    35: 'vastus intermedius',
    36: 'vastus lateralis',
    37: 'vastus medialis',
}

_CMAP = plt.colormaps['hsv'].resampled(37)
LABEL_COLOURS = {lbl: _CMAP(lbl - 1)[:3] for lbl in range(1, 38)}


# ── Helpers ───────────────────────────────────────────────────────────────────

def read_dicom_array(path):
    """Read a greyscale DICOM and return float32 (D, H, W)."""
    ds  = pydicom.dcmread(str(path))
    arr = ds.pixel_array.astype(np.float32)
    arr = arr * float(getattr(ds, 'RescaleSlope', 1.0)) \
            + float(getattr(ds, 'RescaleIntercept', 0.0))
    if arr.ndim == 2:
        arr = arr[np.newaxis]
    return arr


def read_seg_array(path):
    """Read a segmentation DICOM and return int32 labeled volume (D, H, W).

    The file is a Multi-frame Grayscale Secondary Capture (not DICOM SEG).
    Pixel values are label indices spread across 0–255:
        pixel_value = round(label_index × 255 / 37),  label_index ∈ 1–37
    Invert this with:  label_index = round(pixel_value × 37 / 255)
    """
    ds  = pydicom.dcmread(str(path))
    raw = ds.pixel_array.astype(np.float32)   # (D, H, W) or (H, W)
    if raw.ndim == 2:
        raw = raw[np.newaxis]

    # Remap spread-grayscale → 1-37 label indices (0 stays 0)
    labeled = np.round(raw * 37.0 / 255.0).astype(np.int32)
    labeled[raw == 0] = 0
    labeled = np.clip(labeled, 0, 37)
    return labeled


def norm(arr):
    lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
    return np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1).astype(np.float32)


def build_seg_overlay(seg_arr, alpha=0.5):
    """(D, H, W) int32 → (D, H, W, 4) RGBA + legend patches."""
    labels  = sorted(v for v in np.unique(seg_arr) if v != 0)
    rgba    = np.zeros((*seg_arr.shape, 4), dtype=np.float32)
    patches = []
    for lbl in labels:
        colour = LABEL_COLOURS.get(lbl, (1.0, 0.0, 1.0))
        rgba[seg_arr == lbl] = (*colour, alpha)
        name = SHEFFIELD_LABELS.get(lbl, f'?({lbl})')
        patches.append(mpatches.Patch(color=colour, alpha=0.9,
                                      label=f'{lbl}: {name}'))
    return rgba, patches


_cache = {}

def load_sample(idx):
    if idx in _cache:
        return _cache[idx]

    seg_path = os.path.join(SEG_DIR, f'Aug_{idx}_segmentations.dcm')
    img_path = os.path.join(IMG_DIR, f'Aug_{idx}.dcm')

    seg_raw            = read_seg_array(seg_path)
    seg_rgba, patches  = build_seg_overlay(seg_raw)

    img_norm = None
    if os.path.exists(img_path):
        img_norm = norm(read_dicom_array(img_path))

    _cache[idx] = (img_norm, seg_rgba, patches, seg_raw.shape[0])
    return _cache[idx]


print('Helpers ready.')

In [ ]:
# ── Widgets ───────────────────────────────────────────────────────────────────

sample_dd = Dropdown(
    options=INDICES, description='Aug index:',
    layout=widgets.Layout(width='200px'),
)
slice_sl = IntSlider(
    min=0, max=1, step=1, value=0, description='Slice:',
    layout=widgets.Layout(width='600px'),
)
show_seg_cb = Checkbox(
    value=True, description='Show Seg', indent=False,
    layout=widgets.Layout(width='130px'),
)
out = Output()


def render(idx, slice_idx, show_seg):
    if not idx:
        return
    try:
        img_norm, seg_rgba, seg_patches, n_slices = load_sample(idx)
    except Exception as e:
        with out:
            out.clear_output(wait=True)
            print(f'Error loading Aug_{idx}: {e}')
        return

    slice_idx = min(slice_idx, n_slices - 1)
    has_img   = img_norm is not None

    panels = []
    if has_img:
        panels.append(('MRI (Aug image)', img_norm[slice_idx], None, []))
    if show_seg:
        bg = img_norm[slice_idx] if has_img else np.zeros(seg_rgba.shape[1:3])
        panels.append(('Segmentation overlay', bg, seg_rgba[slice_idx], seg_patches))
    if not panels:
        panels.append(('No display selected', np.zeros((64, 64)), None, []))

    n_panels = len(panels)
    fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 7))
    if n_panels == 1:
        axes = [axes]

    for ax, (title, bg_img, overlay, patches) in zip(axes, panels):
        ax.imshow(bg_img, cmap='gray', origin='lower')
        if overlay is not None:
            ax.imshow(overlay, origin='lower')
        if patches:
            ax.legend(handles=patches, loc='lower right', fontsize=7,
                      framealpha=0.7, ncol=2)
        ax.set_title(title, fontsize=11)
        ax.axis('off')

    fig.suptitle(
        f'Aug_{idx}  —  slice {slice_idx}/{n_slices - 1}',
        fontsize=12,
    )
    plt.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()


def _rerender(*_):
    render(sample_dd.value, slice_sl.value, show_seg_cb.value)


def on_sample_change(change):
    if not change['new']:
        return
    try:
        _, _, _, n_slices = load_sample(change['new'])
        slice_sl.max   = n_slices - 1
        slice_sl.value = 0
    except Exception:
        pass
    _rerender()


sample_dd.observe(on_sample_change,         names='value')
slice_sl.observe(lambda _: _rerender(),     names='value')
show_seg_cb.observe(lambda _: _rerender(),  names='value')

# Initial load
if INDICES:
    _, _, _, n0 = load_sample(INDICES[0])
    slice_sl.max = n0 - 1
    render(INDICES[0], 0, show_seg_cb.value)

display(VBox([
    HBox([sample_dd, show_seg_cb]),
    HBox([slice_sl]),
    out,
]))

In [ ]:
# ── Single-muscle viewer ──────────────────────────────────────────────────────
# Pick a muscle from the dropdown; it is highlighted in red on the current slice.
# All other segmentation labels are shown faintly so you can see context.

muscle_options = [f'{lbl}: {name}' for lbl, name in sorted(SHEFFIELD_LABELS.items())]

muscle_dd2  = Dropdown(
    options=muscle_options,
    value='13: gluteus maximus',
    description='Muscle:',
    layout=widgets.Layout(width='340px'),
)
sample_dd2 = Dropdown(
    options=INDICES, description='Aug index:',
    layout=widgets.Layout(width='200px'),
)
slice_sl2 = IntSlider(
    min=0, max=984, step=1, value=0, description='Slice:',
    layout=widgets.Layout(width='600px'),
)
out2 = Output()


def render_muscle(idx, slice_idx, muscle_str):
    if not idx or not muscle_str:
        return
    try:
        img_norm, _, _, n_slices = load_sample(idx)
    except Exception as e:
        with out2:
            out2.clear_output(wait=True)
            print(f'Error: {e}')
        return

    # Re-read the raw labeled array (cached read_seg_array is cheap after first call)
    seg_path = os.path.join(SEG_DIR, f'Aug_{idx}_segmentations.dcm')
    seg_raw  = read_seg_array(seg_path)      # (D, H, W) int32, already cached on disk

    target_lbl = int(muscle_str.split(':')[0])
    slice_idx  = min(slice_idx, n_slices - 1)

    seg_sl  = seg_raw[slice_idx]             # (H, W)
    H, W    = seg_sl.shape

    # Context overlay: all muscles, very faint
    context = np.zeros((H, W, 4), dtype=np.float32)
    for lbl in range(1, 38):
        mask = seg_sl == lbl
        if mask.any():
            c = LABEL_COLOURS.get(lbl, (0.5, 0.5, 0.5))
            context[mask] = (*c, 0.15)

    # Highlight overlay: target muscle in solid red
    highlight = np.zeros((H, W, 4), dtype=np.float32)
    target_mask = seg_sl == target_lbl
    highlight[target_mask] = (1.0, 0.1, 0.1, 0.7)

    present = target_mask.any()
    muscle_name = SHEFFIELD_LABELS.get(target_lbl, str(target_lbl))

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    for ax, (title, overlays) in zip(axes, [
        ('All muscles (context)', [context]),
        (f'{muscle_name}  [label {target_lbl}]', [context, highlight]),
    ]):
        bg = img_norm[slice_idx] if img_norm is not None else np.zeros((H, W))
        ax.imshow(bg, cmap='gray', origin='lower')
        for ov in overlays:
            ax.imshow(ov, origin='lower')
        ax.set_title(title, fontsize=11,
                     color='firebrick' if 'label' in title else 'black')
        ax.axis('off')

    status = 'PRESENT on this slice' if present else 'not on this slice'
    fig.suptitle(
        f'Aug_{idx}  —  slice {slice_idx}/{n_slices - 1}  —  '
        f'{muscle_name}: {status}',
        fontsize=11,
    )
    plt.tight_layout()
    with out2:
        out2.clear_output(wait=True)
        plt.show()


def _rerender2(*_):
    render_muscle(sample_dd2.value, slice_sl2.value, muscle_dd2.value)


def on_sample2_change(change):
    if not change['new']:
        return
    try:
        _, _, _, n_slices = load_sample(change['new'])
        slice_sl2.max   = n_slices - 1
        slice_sl2.value = 0
    except Exception:
        pass
    _rerender2()


sample_dd2.observe(on_sample2_change,        names='value')
muscle_dd2.observe(lambda _: _rerender2(),   names='value')
slice_sl2.observe(lambda _: _rerender2(),    names='value')

# Initial render using same index as top viewer
if INDICES:
    _, _, _, n0 = load_sample(INDICES[0])
    slice_sl2.max = n0 - 1
    render_muscle(INDICES[0], 0, muscle_dd2.value)

display(VBox([
    HBox([sample_dd2, muscle_dd2]),
    HBox([slice_sl2]),
    out2,
]))

In [ ]:
# ── Export current single-muscle view as PNG / TIFF / PDF ────────────────────
# Rebuilds the exact figure from the viewer above using its current widget
# selections (Aug index / Slice / Muscle) and saves it to disk.

_idx    = sample_dd2.value
_slice  = slice_sl2.value
_muscle = muscle_dd2.value

img_norm, _, _, n_slices = load_sample(_idx)
seg_path = os.path.join(SEG_DIR, f'Aug_{_idx}_segmentations.dcm')
seg_raw  = read_seg_array(seg_path)

target_lbl = int(_muscle.split(':')[0])
_slice     = min(_slice, n_slices - 1)
seg_sl     = seg_raw[_slice]
H, W       = seg_sl.shape

context = np.zeros((H, W, 4), dtype=np.float32)
for lbl in range(1, 38):
    mask = seg_sl == lbl
    if mask.any():
        c = LABEL_COLOURS.get(lbl, (0.5, 0.5, 0.5))
        context[mask] = (*c, 0.15)

highlight = np.zeros((H, W, 4), dtype=np.float32)
target_mask = seg_sl == target_lbl
highlight[target_mask] = (1.0, 0.1, 0.1, 0.7)

present     = target_mask.any()
muscle_name = SHEFFIELD_LABELS.get(target_lbl, str(target_lbl))

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, (title, overlays) in zip(axes, [
    ('All muscles (context)', [context]),
    (f'{muscle_name}  [label {target_lbl}]', [context, highlight]),
]):
    bg = img_norm[_slice] if img_norm is not None else np.zeros((H, W))
    ax.imshow(bg, cmap='gray', origin='lower')
    for ov in overlays:
        ax.imshow(ov, origin='lower')
    ax.set_title(title, fontsize=11, color='firebrick' if 'label' in title else 'black')
    ax.axis('off')

status = 'present' if present else 'absent'
fig.suptitle(f'Aug_{_idx}  —  slice {_slice}/{n_slices - 1}  —  {muscle_name}: {status}', fontsize=11)
plt.tight_layout()

safe_muscle = muscle_name.replace(' ', '_')
out_stem = f'sheffield_{_idx}_slice{_slice}_{safe_muscle}'
fig.savefig(f'{out_stem}.png', dpi=200, bbox_inches='tight')
fig.savefig(f'{out_stem}.tiff', dpi=400, bbox_inches='tight')
fig.savefig(f'{out_stem}.pdf', bbox_inches='tight')
plt.show()

print(f'Saved {out_stem}.png / .tiff / .pdf')

In [ ]:
# ── Export two single-panel views as PNG / TIFF / PDF (400 dpi) ──────────────
# Same current widget selections as above.
#   (a) labeled-only : just the highlighted target muscle, no other labels
#   (b) with-context : all other muscles faintly labeled (as in the viewer
#                       above), with the target muscle highlighted on top

_idx    = sample_dd2.value
_slice  = slice_sl2.value
_muscle = muscle_dd2.value

img_norm, _, _, n_slices = load_sample(_idx)
seg_path = os.path.join(SEG_DIR, f'Aug_{_idx}_segmentations.dcm')
seg_raw  = read_seg_array(seg_path)

target_lbl = int(_muscle.split(':')[0])
_slice     = min(_slice, n_slices - 1)
seg_sl     = seg_raw[_slice]
H, W       = seg_sl.shape

context = np.zeros((H, W, 4), dtype=np.float32)
for lbl in range(1, 38):
    mask = seg_sl == lbl
    if mask.any():
        c = LABEL_COLOURS.get(lbl, (0.5, 0.5, 0.5))
        context[mask] = (*c, 0.15)

highlight = np.zeros((H, W, 4), dtype=np.float32)
target_mask = seg_sl == target_lbl
highlight[target_mask] = (1.0, 0.1, 0.1, 0.7)

present     = target_mask.any()
muscle_name = SHEFFIELD_LABELS.get(target_lbl, str(target_lbl))
status      = 'present' if present else 'absent'
bg          = img_norm[_slice] if img_norm is not None else np.zeros((H, W))
safe_muscle = muscle_name.replace(' ', '_')

# (a) labeled-only
fig_a, ax_a = plt.subplots(1, 1, figsize=(7, 7))
ax_a.imshow(bg, cmap='gray', origin='lower')
ax_a.imshow(highlight, origin='lower')
ax_a.set_title(f'{muscle_name}  [label {target_lbl}]', fontsize=11, color='firebrick')
ax_a.axis('off')
fig_a.suptitle(f'Aug_{_idx}  —  slice {_slice}/{n_slices - 1}  —  {muscle_name}: {status}', fontsize=11)
plt.tight_layout()

stem_a = f'sheffield_{_idx}_slice{_slice}_{safe_muscle}_labeled_only'
fig_a.savefig(f'{stem_a}.png', dpi=400, bbox_inches='tight')
fig_a.savefig(f'{stem_a}.tiff', dpi=400, bbox_inches='tight')
fig_a.savefig(f'{stem_a}.pdf', dpi=400, bbox_inches='tight')
plt.show()

# (b) with-context
fig_b, ax_b = plt.subplots(1, 1, figsize=(7, 7))
ax_b.imshow(bg, cmap='gray', origin='lower')
ax_b.imshow(context, origin='lower')
ax_b.imshow(highlight, origin='lower')
ax_b.set_title(f'{muscle_name}  [label {target_lbl}]', fontsize=11, color='firebrick')
ax_b.axis('off')
fig_b.suptitle(f'Aug_{_idx}  —  slice {_slice}/{n_slices - 1}  —  {muscle_name}: {status}', fontsize=11)
plt.tight_layout()

stem_b = f'sheffield_{_idx}_slice{_slice}_{safe_muscle}_with_context'
#fig_b.savefig(f'{stem_b}.png', dpi=400, bbox_inches='tight')
#fig_b.savefig(f'{stem_b}.tiff', dpi=400, bbox_inches='tight')
#fig_b.savefig(f'{stem_b}.pdf', dpi=400, bbox_inches='tight')
plt.show()

#print(f'Saved {stem_a}.png / .tiff / .pdf (400 dpi)')
#print(f'Saved {stem_b}.png / .tiff / .pdf (400 dpi)')

In [ ]:
# ── Export one side-by-side image (labeled-only | with-context), 400 dpi ─────
# Same current widget selections as above. Left panel: only the highlighted
# target muscle, no other labels. Right panel: all other muscles faintly
# labeled (as in the viewer above), with the target muscle highlighted on top.
# Both panels are saved together as a single PNG / TIFF / PDF.

_idx    = sample_dd2.value
_slice  = slice_sl2.value
_muscle = muscle_dd2.value

img_norm, _, _, n_slices = load_sample(_idx)
seg_path = os.path.join(SEG_DIR, f'Aug_{_idx}_segmentations.dcm')
seg_raw  = read_seg_array(seg_path)

target_lbl = int(_muscle.split(':')[0])
_slice     = min(_slice, n_slices - 1)
seg_sl     = seg_raw[_slice]
H, W       = seg_sl.shape

context = np.zeros((H, W, 4), dtype=np.float32)
for lbl in range(1, 38):
    mask = seg_sl == lbl
    if mask.any():
        c = LABEL_COLOURS.get(lbl, (0.5, 0.5, 0.5))
        context[mask] = (*c, 0.15)

highlight = np.zeros((H, W, 4), dtype=np.float32)
target_mask = seg_sl == target_lbl
highlight[target_mask] = (1.0, 0.1, 0.1, 0.7)

present     = target_mask.any()
muscle_name = SHEFFIELD_LABELS.get(target_lbl, str(target_lbl))
status      = 'present' if present else 'absent'
bg          = img_norm[_slice] if img_norm is not None else np.zeros((H, W))
safe_muscle = muscle_name.replace(' ', '_')

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(14, 7))

ax_a.imshow(bg, cmap='gray', origin='lower')
ax_a.imshow(highlight, origin='lower')
ax_a.set_title(f'{muscle_name}  [label {target_lbl}]  (no context)', fontsize=11, color='firebrick')
ax_a.axis('off')

ax_b.imshow(bg, cmap='gray', origin='lower')
ax_b.imshow(context, origin='lower')
ax_b.imshow(highlight, origin='lower')
ax_b.set_title(f'{muscle_name}  [label {target_lbl}]  (with context)', fontsize=11, color='firebrick')
ax_b.axis('off')

fig.suptitle(f'Aug_{_idx}  —  slice {_slice}/{n_slices - 1}  —  {muscle_name}: {status}', fontsize=11)
plt.tight_layout()

out_stem = f'sheffield_{_idx}_slice{_slice}_{safe_muscle}_side_by_side'
fig.savefig(f'{out_stem}.png', dpi=400, bbox_inches='tight')
fig.savefig(f'{out_stem}.tiff', dpi=400, bbox_inches='tight')
fig.savefig(f'{out_stem}.pdf', dpi=400, bbox_inches='tight')
plt.show()

print(f'Saved {out_stem}.png / .tiff / .pdf (400 dpi)')